<a href="https://colab.research.google.com/github/2470024/data-science-portfolio/blob/main/AirLab_SIMPLE_Purifier_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌬️ 空気の流れと空気清浄機をシミュレーションしてみよう（CFD体験）

このプログラムは、**部屋の中の空気の流れ**と  
**汚れた空気（濃度）が時間とともにどう動くか**を  
コンピュータで再現（シミュレーション）するものです。

専門的には **CFD（数値流体力学）** と呼ばれますが、  
ここでは「空気の動きを計算で追いかける実験」だと思ってください。

---

## 🏠 このシミュレーションで何をしているの？

部屋の中には、次のものがあります。

- ✅ 天井からゆっくり空気が入ってくる
- ✅ ベッド（空気が通れない物体）
- ✅ 空気清浄機（風を出し、汚れを減らす）
- ✅ 汚染源（におい・ウイルス・煙のイメージ）
- ✅ ある1点で「空気の汚れ」を観測

コンピュータの中で部屋を**小さなマス目（3次元）**に分けて、
それぞれの場所で

- 空気の速さ
- 空気の向き
- 汚れの量

を少しずつ時間を進めながら計算しています。

---

## 🌀 空気の流れ（CFDパート）

- 空気は「速さ（u, v, w）」を持っています  
- ベッドの中では空気は止まります
- 天井から下向きに空気が入ります
- 空気清浄機の中では **体積力** という方法で  
  「ファンが押す力」を与えています

💡 **ポイント**  
羽根を回す代わりに  
「この場所では空気が押される」と計算するのが  
**体積力ファンモデル**です。

---

## ☁️ 汚れ（濃度）の動き

汚れは次のルールで動きます。

1. 空気の流れに乗って運ばれる（風で流れる）
2. 少しずつ周囲に広がる（拡散）
3. 汚染源から一定時間だけ発生する
4. 空気清浄機で一部が除去される
5. 壁や床では外に出ない

これにより、
「時間がたつと汚れがどう減るか」
を観察できます。

---

## 📊 結果の見方

- 青い矢印：空気の流れ
- 色のついたモヤ：汚れの濃度
- 赤い点：汚染源
- 青い点：観測点
- 緑の線：ベッド
- 水色の線：空気清浄機

最後に表示されるグラフは  
**観測点の汚れが時間とともにどう変わったか**を示しています。

---

# ✏️ 演習課題（やってみよう）

## 📝 演習1：空気清浄機の強さを変えてみよう
**目的**：清浄機のパワーが強いとどうなるか調べる

1. `fan_strength` を次のように変えて実行せよ  
   - 0.0  
   - 1.0  
   - 2.0
2. 観測点の最終濃度を比べなさい

👉 **考察**  
- 強くすると必ず空気はきれいになる？
- 流れが強すぎるとどうなる？

---

## 📝 演習2：汚染源の場所を変えてみよう
**目的**：汚れが出る場所で結果がどう変わるか調べる

1. `SOURCE["pos"]` の高さ（z）を  
   - 0.3  
   - 1.0  
   - 2.0  
   に変えて実行せよ
2. 観測点の濃度の時間変化を比べなさい

👉 **考察**  
- 天井に近い方が危険？
- 空気の流れとの関係は？

---

## 📝 演習3：観測点を移動させてみよう
**目的**：「どこが一番安全か」を探す

1. `obs_point` を部屋のいくつかの位置に変える
2. 最終濃度が最も小さい場所を見つけなさい

👉 **発展問題**  
- 「病室」「教室」なら観測点はどこがよい？

---

## 🎯 まとめ

このプログラムは、

- 物理（空気の流れ）
- 数学（差分計算）
- 情報（シミュレーション）
- 社会（感染対策・環境）

がつながった **データサイエンスの実例**です。

「現実では試せないこと」を  
**コンピュータの中で安全に試す**  
それがシミュレーションの力です 🚀


In [ ]:
# ============================================================
# 🎓 学生配布用：体積力型ファンモデル CFD（Colab対応）
# ============================================================

import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# ============================================================
# 🧪 実験パラメータ（学生が変更するのはここだけ）
# ============================================================

EXPERIMENT = {
    "U_in": 0.1,                  # 天井流入風速 [m/s]
    "fan_strength": 2.0,           # ⭐ 清浄機強度（0.0〜2.0）
    "fan_x": 1.0,                  # 清浄機 X 位置
    "fan_y": 3.0,                  # 清浄機 Y 位置
    "D": 5e-4,                     # 拡散係数
    "nsteps":9000,                # 濃度輸送ステップ
    "obs_point": (2.5, 2.0, 1.5)   # 観測点
}

SOURCE = {
    "pos": (2.5, 2.0, 0.3),   # 発生位置
    "emit_steps": 300,        # ⭐ 発生する時間長さ（タイムステップ数）
    "strength": 10          # ⭐ 1ステップ当たりの注入量
}

# ============================================================
# ⚙ CFD 設定（変更しない）
# ============================================================

Nx, Ny, Nz = 20, 20, 15
Lx, Ly, Lz = 4.0, 4.0, 3.0
nu = 1e-5
rho = 1.0
dt = 0.002
max_iter = 200

# ============================================================
# 📐 格子生成
# ============================================================

x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)
z = np.linspace(0, Lz, Nz)
dx, dy, dz = x[1]-x[0], y[1]-y[0], z[1]-z[0]

Xc, Yc, Zc = np.meshgrid(x, y, z, indexing="ij")

# ============================================================
# 🛏 固体（ベッド）
# ============================================================

solid = np.zeros_like(Xc, dtype=bool)

solid |= (
    (1.0 <= Xc) & (Xc <= 3.0) &
    (1.0 <= Yc) & (Yc <= 2.0) &
    (0.0 <= Zc) & (Zc <= 1.0)
)

fluid = ~solid

# ============================================================
# 🌀 初期化
# ============================================================

u = np.zeros_like(Xc)
v = np.zeros_like(Xc)
w = np.zeros_like(Xc)
p = np.zeros_like(Xc)

# ============================================================
# 🔢 微分演算
# ============================================================

def grad(f):
    return (
        (np.roll(f,-1,0)-np.roll(f,1,0))/(2*dx),
        (np.roll(f,-1,1)-np.roll(f,1,1))/(2*dy),
        (np.roll(f,-1,2)-np.roll(f,1,2))/(2*dz)
    )

def div(u,v,w):
    du,_,_ = grad(u)
    _,dv,_ = grad(v)
    _,_,dw = grad(w)
    return du + dv + dw

def lap(f):
    return (
        (np.roll(f,-1,0)-2*f+np.roll(f,1,0))/dx**2 +
        (np.roll(f,-1,1)-2*f+np.roll(f,1,1))/dy**2 +
        (np.roll(f,-1,2)-2*f+np.roll(f,1,2))/dz**2
    )

# ============================================================
# 🧩 SIMPLE + 体積力ファンモデル
# ============================================================

print("=== CFD (SIMPLE + Fan Force) ===")

# 清浄機（体積力がかかる領域）
fan_region = (
    (abs(Xc - EXPERIMENT["fan_x"]) < 0.4) &
    (abs(Yc - EXPERIMENT["fan_y"]) < 0.4) &
    (0.3 < Zc) & (Zc < 1.5)
)

F_base = 0.2  # 基本ファン力 [m/s²]

for it in range(max_iter):

    # 天井流入
    inlet = (
        (1.5 < Xc) & (Xc < 2.5) &
        (1.5 < Yc) & (Yc < 2.5) &
        (np.isclose(Zc, Lz))
    )
    w[inlet] = EXPERIMENT["U_in"]

    # ---------
    # 体積力（ファン）
    # ---------
    Fx = np.zeros_like(Xc)
    Fy = np.zeros_like(Xc)
    Fz = np.zeros_like(Xc)

    Fx[fan_region] = F_base * EXPERIMENT["fan_strength"]

    # 速度予測
    du_adv = u*grad(u)[0] + v*grad(u)[1] + w*grad(u)[2]
    dv_adv = u*grad(v)[0] + v*grad(v)[1] + w*grad(v)[2]
    dw_adv = u*grad(w)[0] + v*grad(w)[1] + w*grad(w)[2]

    u_star = u + dt*(-du_adv + nu*lap(u) + Fx)
    v_star = v + dt*(-dv_adv + nu*lap(v) + Fy)
    w_star = w + dt*(-dw_adv + nu*lap(w) + Fz)

    u_star[solid] = v_star[solid] = w_star[solid] = 0.0

    # 圧力補正（簡易）
    p -= 0.3 * div(u_star, v_star, w_star)
    dpdx, dpdy, dpdz = grad(p)

    u = u_star - dt*dpdx
    v = v_star - dt*dpdy
    w = w_star - dt*dpdz

    if it % 20 == 0:
        d = np.sqrt(np.mean(div(u,v,w)[fluid]**2))
        print(f"Iter {it}, div L2 = {d:.2e}")

# ============================================================
# ☁ 濃度輸送
# ============================================================

# ============================================================
# ☁ 濃度輸送 + フィルター付き清浄機
# ============================================================

C = np.zeros_like(Xc)

# --- 汚染源 ---
sx, sy, sz = SOURCE["pos"]   # ⭐ ここで SOURCE から受け取る
ix = np.argmin(abs(x-sx))
iy = np.argmin(abs(y-sy))
iz = np.argmin(abs(z-sz))
#C[ix,iy,iz] = 1.0

# --- 観測点 ---
ox, oy, oz = EXPERIMENT["obs_point"]
iox = np.argmin(abs(x-ox))
ioy = np.argmin(abs(y-oy))
ioz = np.argmin(abs(z-oz))

# --- ⭐ フィルター設定 ⭐ ---
eta = 0.1   # フィルター効率（0〜1）
# 吸込み点（ファン中心）
f_ix = np.argmin(abs(x - EXPERIMENT["fan_x"]))
f_iy = np.argmin(abs(y - EXPERIMENT["fan_y"]))
f_iz = np.argmin(abs(z - 0.8))

# 吐出口（少し前方へ）
out_ix = np.argmin(abs(x - (EXPERIMENT["fan_x"] + 0.3)))
out_iy = f_iy
out_iz = f_iz

C_hist = []

for step in range(EXPERIMENT["nsteps"]):

    # ---- 基本輸送 ----
    dCx, dCy, dCz = grad(C)
    adv = -(u*dCx + v*dCy + w*dCz)
    diff = EXPERIMENT["D"] * lap(C)
    C += dt*(adv + diff)

    # =========================================
    # ⭐ 汚染源：一定時間だけ発生 ⭐
    # =========================================
    if step < SOURCE["emit_steps"]:
    #    C[ix, iy, iz] += SOURCE["strength"]
         C[ix, iy, iz] = SOURCE["strength"]
    # ---- ⭐ 天井吸い取り（安定化処理）⭐ ----
    #C[:, :, -1] = 0.0

    # ---------------------------
    # ⭐ 壁のゼロフラックス境界 条件（周期の除去）
    # ---------------------------
    # X方向壁
    C[0, :, :]  = C[1, :, :]
    C[-1, :, :] = C[-2, :, :]

    # Y方向壁
    C[:, 0, :]  = C[:, 1, :]
    C[:, -1, :] = C[:, -2, :]

    # Z方向壁（床・天井）
    C[:, :, 0]  = C[:, :, 1]
    C[:, :, -1] = 0.0      # ⭐ 天井吸着モデルの場合は吸い取り


    # ---- ⭐ フィルター付き清浄機 ⭐ ----
    Cin = C[f_ix, f_iy, f_iz]
    Cout = (1 - eta) * Cin

    # 吸い込み点 → 少し減らす（完全ゼロにしない）
    C[f_ix, f_iy, f_iz] *= (1 - 0.3*eta)

    # 吐出口へ排出
    C[out_ix, out_iy, out_iz] = max(Cout, 0.0)

    # ---- 壁条件 ----
    C[solid] = 0
    # 安定化リミッタ
    C = np.clip(C, 0, 1.0)
    C[C < 0] = 0
    C[0,:,:] = 0
    C[-1,:,:] = 0
    C[:,0,:] = 0
    C[:,-1,:] = 0
    # ---- わずかな自然減衰 ----
    C *= 0.999

    # ---- 観測履歴 ----
    C_hist.append(C[iox, ioy, ioz])


# ============================================================
# 📊 可視化
# ============================================================

U_mag = np.sqrt(u**2 + v**2 + w**2)
mask = (U_mag > 1e-4) & fluid

cone = go.Cone(
    x=Xc[mask], y=Yc[mask], z=Zc[mask],
    u=u[mask], v=v[mask], w=w[mask],
    sizeref=0.5,
    colorscale="Blues",
    anchor="tail",
    name="Velocity",
    showscale=False,    # ← これも非表示
    showlegend=False
)

volume = go.Volume(
    x=Xc.flatten(), y=Yc.flatten(), z=Zc.flatten(),
    value=C.flatten(),
    isomin=0.001*C.max(),
    isomax=C.max(),
    opacity=0.12,
    surface_count=30,
    colorscale="Viridis",
    showscale=False,   # ← カラーバー非表示
    showlegend=False   # ← レジェンド非表示
)

fig = go.Figure(data=[volume, cone])
fig.update_layout(
    title="3D CFD with Volume-Force Fan Model",
    scene=dict(
        xaxis_title="X [m]",
        yaxis_title="Y [m]",
        zaxis_title="Z [m]"
    )
)

# ============================================================
# 追加可視化（汚染源・観測点・ベッド線画・清浄機線画）
# ============================================================

# -----------------------------------------
# ① 汚染源（赤マーカー）
# -----------------------------------------
fig.add_trace(go.Scatter3d(
    x=[sx], y=[sy], z=[sz],
    mode="markers",
    marker=dict(size=3, color="red"),
    name="Source"
))

# -----------------------------------------
# ② 観測点（青マーカー）
# -----------------------------------------
fig.add_trace(go.Scatter3d(
    x=[ox], y=[oy], z=[oz],
    mode="markers",
    marker=dict(size=6, color="blue"),
    name="Observation"
))

# -----------------------------------------
# ③ ベッド（線画）
# -----------------------------------------
bed_x1, bed_x2 = 1.0, 3.0
bed_y1, bed_y2 = 1.0, 2.0
bed_h = 1.0

bed_lines = [
    # bottom
    (bed_x1, bed_y1, 0, bed_x2, bed_y1, 0),
    (bed_x2, bed_y1, 0, bed_x2, bed_y2, 0),
    (bed_x2, bed_y2, 0, bed_x1, bed_y2, 0),
    (bed_x1, bed_y2, 0, bed_x1, bed_y1, 0),

    # top
    (bed_x1, bed_y1, bed_h, bed_x2, bed_y1, bed_h),
    (bed_x2, bed_y1, bed_h, bed_x2, bed_y2, bed_h),
    (bed_x2, bed_y2, bed_h, bed_x1, bed_y2, bed_h),
    (bed_x1, bed_y2, bed_h, bed_x1, bed_y1, bed_h),

    # pillars
    (bed_x1, bed_y1, 0, bed_x1, bed_y1, bed_h),
    (bed_x2, bed_y1, 0, bed_x2, bed_y1, bed_h),
    (bed_x2, bed_y2, 0, bed_x2, bed_y2, bed_h),
    (bed_x1, bed_y2, 0, bed_x1, bed_y2, bed_h),
]

for (x1,y1,z1,x2,y2,z2) in bed_lines:
    fig.add_trace(go.Scatter3d(
        x=[x1,x2], y=[y1,y2], z=[z1,z2],
        mode="lines",
        line=dict(color="green", width=6),
        name="Bed"
    ))

# -----------------------------------------
# ④ 清浄機（線画）
# -----------------------------------------
px = EXPERIMENT["fan_x"]
py = EXPERIMENT["fan_y"]

fan_px = EXPERIMENT["fan_x"]
fan_py = EXPERIMENT["fan_y"]

px1, px2 = fan_px-0.4, fan_px+0.4
py1, py2 = fan_py-0.4, fan_py+0.4

ph = 1.5

purifier_edges = [
    # bottom
    (px1,py1,0, px2,py1,0),
    (px2,py1,0, px2,py2,0),
    (px2,py2,0, px1,py2,0),
    (px1,py2,0, px1,py1,0),

    # top
    (px1,py1,ph, px2,py1,ph),
    (px2,py1,ph, px2,py2,ph),
    (px2,py2,ph, px1,py2,ph),
    (px1,py2,ph, px1,py1,ph),

    # sides
    (px1,py1,0, px1,py1,ph),
    (px2,py1,0, px2,py1,ph),
    (px2,py2,0, px2,py2,ph),
    (px1,py2,0, px1,py2,ph),
]

for (x1,y1,z1,x2,y2,z2) in purifier_edges:
    fig.add_trace(go.Scatter3d(
        x=[x1,x2], y=[y1,y2], z=[z1,z2],
        mode="lines",
        line=dict(color="cyan", width=6),
        name="Purifier"
    ))


fig.show()

import plotly.express as px  # 念のため

px.line(
    x=np.arange(len(C_hist)),
    y=C_hist,
    labels={"x": "Time step", "y": "Concentration"},
    title="Observation Point Concentration"
).show()



# ============================================================
# 🎯 目的変数：観測点の最終濃度（出力）
# ============================================================

# 観測点（実座標 → 格子インデックス変換）
ox, oy, oz = EXPERIMENT["obs_point"]

ix = np.argmin(np.abs(x - ox))
iy = np.argmin(np.abs(y - oy))
iz = np.argmin(np.abs(z - oz))

# 最終濃度（目的変数）
final_concentration = C[ix, iy, iz]

print("\n==============================")
print(" 🎯 目的変数（最終濃度）")
print(f" 観測点 : (x={ox:.2f}, y={oy:.2f}, z={oz:.2f}) [m]")
print(f" 最終濃度 : {final_concentration:.6e}")   # ★指数表記
print("==============================\n")

=== CFD (SIMPLE + Fan Force) ===
Iter 0, div L2 = 1.48e-02
Iter 20, div L2 = 9.21e-03
Iter 40, div L2 = 1.07e-02
Iter 60, div L2 = 1.02e-02
Iter 80, div L2 = 1.08e-02
Iter 100, div L2 = 1.07e-02
Iter 120, div L2 = 1.14e-02
Iter 140, div L2 = 1.10e-02
Iter 160, div L2 = 1.17e-02
Iter 180, div L2 = 1.18e-02



 🎯 目的変数（最終濃度）
 観測点 : (x=2.50, y=2.00, z=1.50) [m]
 最終濃度 : 8.734487e-12



In [ ]:
import numpy as np
import pandas as pd

# ================================
# 1️⃣ パラメータ読み込み
# ================================
# 例：列名は以下を想定
# fan_strength , fan_x , fan_y , D , emit_steps
params = pd.read_csv("parameter_sets.csv")
display(params)   # ← テーブル確認（Notebook表示用）


# ================================
# 2️⃣ ここに既存の CFD モデル設定
#     （必要ならサイズ変更OK）
# ================================
Nx, Ny, Nz = 20, 20, 15
Lx, Ly, Lz = 4.0, 4.0, 3.0
dt = 0.01

x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)
z = np.linspace(0, Lz, Nz)
dx, dy, dz = x[1]-x[0], y[1]-y[0], z[1]-z[0]
Xc, Yc, Zc = np.meshgrid(x, y, z, indexing="ij")


# ================================
# ⭐ CFD + 濃度モデル関数
# ================================
def run_cfd_case(fan_strength, fan_x, fan_y, D, emit_steps):
    """
    与えられたパラメータで CFD を1回実行し
    観測点の最終濃度を返す
    """

    # --- フィールド初期化 ---
    u = np.zeros_like(Xc)
    v = np.zeros_like(Xc)
    w = np.zeros_like(Xc)
    C = np.zeros_like(Xc)

    # --- ファンによる強制流れ（簡易モデル） ---
    fan_region = (
        (abs(Xc - fan_x) < 0.4) &
        (abs(Yc - fan_y) < 0.4) &
        (0.3 < Zc) & (Zc < 1.5)
    )

    # --- 観測点 ---
    obs = (2.5, 2.0, 1.5)
    ix = np.argmin(abs(x-obs[0]))
    iy = np.argmin(abs(y-obs[1]))
    iz = np.argmin(abs(z-obs[2]))

    # --- 汚染源 ---
    src = (2.0, 2.0, 0.3)
    sx = np.argmin(abs(x-src[0]))
    sy = np.argmin(abs(y-src[1]))
    sz = np.argmin(abs(z-src[2]))

    # -----------------------------
    # 時間発展
    # -----------------------------
    for step in range(2000):

        # ⭐ 一定時間だけ発生 ⭐
        if step < emit_steps:
            C[sx, sy, sz] = 1.0

        # --- ファン：風速付与 ---
        u[fan_region] = fan_strength * 0.2

        # --- 濃度移流拡散 ---
        dCx = (np.roll(C,-1,0)-np.roll(C,1,0))/(2*dx)
        dCy = (np.roll(C,-1,1)-np.roll(C,1,1))/(2*dy)
        dCz = (np.roll(C,-1,2)-np.roll(C,1,2))/(2*dz)

        adv = -(u*dCx + v*dCy + w*dCz)

        lap = (
            (np.roll(C,-1,0)-2*C+np.roll(C,1,0))/dx**2 +
            (np.roll(C,-1,1)-2*C+np.roll(C,1,1))/dy**2 +
            (np.roll(C,-1,2)-2*C+np.roll(C,1,2))/dz**2
        )

        C += dt*(adv + D*lap)

        # 負値防止
        C[C < 0] = 0.0

    # 🎯 観測点の最終値を返す
    return float(C[ix,iy,iz])


# ================================
# 3️⃣ 全ケースをまとめて実行！
# ================================
results = []

for i, row in params.iterrows():

    result = run_cfd_case(
        fan_strength = row["fan_strength"],
        fan_x        = row["fan_x"],
        fan_y        = row["fan_y"],
        D            = row["D"],
        emit_steps   = int(row["emit_steps"])
    )

    results.append(result)
    print(f"Case {i+1} Done → Final Concentration = {result:.3e}")


# ================================
# 4️⃣ 結果をテーブル化
# ================================
params["Final_Concentration"] = results
display(params)

# 必要なら CSV 保存
params.to_csv("simulation_results.csv", index=False)

print("\n🎉 すべてのケース計算が完了しました！")
print("結果は simulation_results.csv に保存されました。")


,No,fan_strength,fan_x,fan_y,D,emit_steps
0,1,3.788116,2.895904,1.656523,0.0005,10000
1,2,4.690300,1.475763,2.155422,0.0005,10000
2,3,4.712630,1.844586,1.028659,0.0005,10000
3,4,3.236096,1.615869,1.499746,0.0005,10000
4,5,4.271686,2.100006,1.441975,0.0005,10000
5,6,8.204270,1.543902,2.179577,0.0005,10000
6,7,9.893350,2.043016,1.874942,0.0005,10000
7,8,6.877758,2.037912,2.164834,0.0005,10000
8,9,8.653643,2.664167,1.760494,0.0005,10000
9,10,6.742159,2.627217,2.401241,0.0005,10000


Case 1 Done → Final Concentration = 1.264e-09
Case 2 Done → Final Concentration = 1.229e-09
Case 3 Done → Final Concentration = 1.229e-09
Case 4 Done → Final Concentration = 1.229e-09
Case 5 Done → Final Concentration = 1.229e-09
Case 6 Done → Final Concentration = 1.224e-09
Case 7 Done → Final Concentration = 3.724e-10
Case 8 Done → Final Concentration = 5.595e-10
Case 9 Done → Final Concentration = 5.308e-08
Case 10 Done → Final Concentration = 5.249e-08
Case 11 Done → Final Concentration = 1.229e-09
Case 12 Done → Final Concentration = 1.229e-09
Case 13 Done → Final Concentration = 1.224e-09
Case 14 Done → Final Concentration = 1.229e-09
Case 15 Done → Final Concentration = 1.229e-09
Case 16 Done → Final Concentration = 6.748e-10
Case 17 Done → Final Concentration = 1.229e-09
Case 18 Done → Final Concentration = 1.229e-09
Case 19 Done → Final Concentration = 1.224e-09
Case 20 Done → Final Concentration = 1.692e-09
Case 21 Done → Final Concentration = 1.224e-09
Case 22 Done → Final C

,No,fan_strength,fan_x,fan_y,D,emit_steps,Final_Concentration
0,1,3.788116,2.895904,1.656523,0.0005,10000,1.263654e-09
1,2,4.690300,1.475763,2.155422,0.0005,10000,1.228728e-09
2,3,4.712630,1.844586,1.028659,0.0005,10000,1.228736e-09
3,4,3.236096,1.615869,1.499746,0.0005,10000,1.228678e-09
4,5,4.271686,2.100006,1.441975,0.0005,10000,1.228744e-09
5,6,8.204270,1.543902,2.179577,0.0005,10000,1.224430e-09
6,7,9.893350,2.043016,1.874942,0.0005,10000,3.724371e-10
7,8,6.877758,2.037912,2.164834,0.0005,10000,5.595050e-10
8,9,8.653643,2.664167,1.760494,0.0005,10000,5.307638e-08
9,10,6.742159,2.627217,2.401241,0.0005,10000,5.248635e-08



🎉 すべてのケース計算が完了しました！
結果は simulation_results.csv に保存されました。
